In [1]:
# put this at the top of every notebook
import plotly.io as pio
pio.renderers.default = "notebook_connected"

In [2]:
# Cell 1 — setup
from yfinance_api3.classes.stock_client import StockClient
from yfinance_api3.classes.quant_analytics import QuantAnalytics
from yfinance_api3.classes.options import OptionsAnalyzer
import yfinance_api3.modules.plots as plots

client = StockClient()
quant  = QuantAnalytics(client)
opt    = OptionsAnalyzer(client, "SPY")

In [3]:
# Cell 2 — expiries
print(opt.expiries())


['2026-04-30', '2026-05-01', '2026-05-04', '2026-05-05', '2026-05-06', '2026-05-07', '2026-05-08', '2026-05-15', '2026-05-22', '2026-05-29', '2026-06-05', '2026-06-18', '2026-06-30', '2026-07-17', '2026-07-31', '2026-08-21', '2026-08-31', '2026-09-18', '2026-09-30', '2026-12-18', '2026-12-31', '2027-01-15', '2027-03-19', '2027-03-31', '2027-06-17', '2027-09-17', '2027-12-17', '2028-01-21', '2028-06-16', '2028-12-15']


In [4]:
# Cell 3 — front month chain
expiry = opt.nearest_expiry(0)
print(f"Front month: {expiry}")
df = opt.chain(expiry)
print(df.head(20))

Front month: 2026-04-30
        expiry  type  strike     bid     ask  last_price  volume  \
0   2026-04-30  call   430.0  283.92  286.67      246.20     1.0   
1   2026-04-30   put   430.0    0.00    0.01        0.01    27.0   
2   2026-04-30   put   435.0    0.00    0.01        0.01     1.0   
3   2026-04-30  call   435.0  278.88  281.68      277.69     2.0   
4   2026-04-30  call   440.0  273.88  276.68      267.50     0.0   
5   2026-04-30   put   440.0    0.00    0.01        0.01     4.0   
6   2026-04-30   put   445.0    0.00    0.01        0.02     8.0   
7   2026-04-30  call   445.0  268.95  271.76      262.60     2.0   
8   2026-04-30  call   450.0  259.53  262.34      228.53     0.0   
9   2026-04-30   put   450.0    0.00    0.01        0.01     1.0   
10  2026-04-30   put   455.0    0.00    0.01        0.01     7.0   
11  2026-04-30   put   460.0    0.00    0.01        0.01     3.0   
12  2026-04-30  call   460.0  249.53  252.35      198.10     1.0   
13  2026-04-30   put   4

In [5]:
# Cell 4 — summary
print(opt.summary(expiry))

{'symbol': 'SPY', 'expiry': '2026-04-30', 'days_to_expiry': 0, 'n_strikes': 223, 'total_call_oi': 490466, 'total_put_oi': 954324, 'total_call_vol': 1973209, 'total_put_vol': 2159120, 'put_call_ratio_oi': 1.946, 'put_call_ratio_vol': 1.094, 'max_pain_strike': 699.0, 'max_oi_call_strike': np.float64(725.0), 'max_oi_put_strike': np.float64(655.0), 'avg_call_iv': 0.1652, 'avg_put_iv': 0.8886, 'iv_skew': 0.7233}


In [6]:
# Cell 5 — max pain (knockout price)
print(f"Max pain: ${opt.max_pain(expiry):,.2f}")

Max pain: $699.00


In [7]:
# Cell 6 — put/call ratio
print(opt.put_call_ratio())

        expiry  days_to_expiry  call_oi   put_oi  put_call_ratio
0   2026-04-30               0   490466   954324           1.946
1   2026-05-01               1   340670   944088           2.771
2   2026-05-04               4    69547   107664           1.548
3   2026-05-05               5    15612    26674           1.709
4   2026-05-06               6     9906    16562           1.672
5   2026-05-07               7     7065    11699           1.656
6   2026-05-08               8   195246   520927           2.668
7   2026-05-15              15   713075  2773454           3.889
8   2026-05-22              22    90032   509945           5.664
9   2026-05-29              29   225686  1247417           5.527
10  2026-06-05              36    13348    26087           1.954
11  2026-06-18              49   819776  1945924           2.374
12  2026-06-30              61   158136   293771           1.858
13  2026-07-17              78   170525   260807           1.529
14  2026-07-31           

In [8]:
# Cell 7 — plots
plots.options_chain(opt, expiry).show()
plots.options_oi_profile(opt, expiry).show()
plots.options_put_call(opt).show()
plots.options_surface(opt, max_expiries=6).show()

In [9]:
# front 12 expiries (~1 year for weeklies, or full year for monthlies)
plots.options_max_pain(opt, max_expiries=12).show()

# more expiries
plots.options_max_pain(opt, max_expiries=24).show()

In [10]:
from yfinance_api3.classes.options import OptionsAnalyzer
import yfinance_api3.modules.plots as plots

opt = OptionsAnalyzer(client, "SPY")

# Greeks
df = opt.greeks(opt.nearest_expiry(0))
print("Greeks columns:", df.columns.tolist())
print(df[["type","strike","delta","gamma","theta","vega","rho","greek_source"]].head(10))

# GEX
strike_gex = opt.gex_by_strike()
by_exp     = opt.gex_by_expiry()
total      = opt.gex_total()

flip     = total.get("flip_strike")
flip_str = f"${flip:,.2f}" if flip is not None else "none detected"

print(f"\nRegime:          {total['regime_label']}")
print(f"Total GEX:       ${total['total_net_gex']/1e6:.1f}M")
print(f"GEX flip:        {flip_str}")
print(f"Dominant expiry: {total.get('dominant_expiry', '—')}")

# Unusual activity
unusual = opt.unusual_activity(vol_oi_threshold=3.0, min_volume=100)
print(f"\nUnusual activity — {len(unusual)} strikes flagged")
if not unusual.empty:
    print(unusual[["type","strike","expiry","volume",
                   "open_interest","vol_oi_ratio","signal"]].head(10))

# Plots
plots.options_chain(opt, opt.nearest_expiry(0)).show()
plots.options_oi_profile(opt, opt.nearest_expiry(0)).show()
plots.options_put_call(opt, by="oi").show()
plots.options_surface(opt, max_expiries=6).show()
plots.options_max_pain(opt, max_expiries=12).show()
plots.options_gex(opt, max_expiries=12).show()
plots.options_unusual(opt, vol_oi_threshold=3.0, min_volume=100).show()

Greeks columns: ['expiry', 'type', 'strike', 'bid', 'ask', 'last_price', 'volume', 'open_interest', 'implied_volatility', 'in_the_money', 'contract', 'delta', 'gamma', 'theta', 'vega', 'rho', 'theoretical_price', 'greek_source']
   type  strike   delta     gamma   theta    vega     rho   greek_source
0  call   430.0  1.0000  0.000000 -0.0589  0.0000  0.0118  black_scholes
1   put   430.0 -0.0002  0.000007 -0.0393  0.0003 -0.0000  black_scholes
2   put   435.0 -0.0002  0.000007 -0.0389  0.0003 -0.0000  black_scholes
3  call   435.0  1.0000  0.000000 -0.0596  0.0000  0.0119  black_scholes
4  call   440.0  1.0000  0.000000 -0.0603  0.0000  0.0121  black_scholes
5   put   440.0 -0.0002  0.000008 -0.0383  0.0003 -0.0000  black_scholes
6   put   445.0 -0.0002  0.000008 -0.0376  0.0003 -0.0000  black_scholes
7  call   445.0  0.9995  0.000017 -0.1599  0.0007  0.0122  black_scholes
8  call   450.0  1.0000  0.000000 -0.0616  0.0000  0.0123  black_scholes
9   put   450.0 -0.0002  0.000008 -0.0367

In [11]:
total = opt.gex_total()
print(total.keys())

dict_keys(['symbol', 'total_call_gex', 'total_put_gex', 'total_net_gex', 'regime', 'regime_label', 'flip_strike', 'dominant_expiry'])


In [12]:
by_exp = opt.gex_by_expiry()
print(by_exp.columns.tolist())
print(by_exp.head(2))

['expiry', 'days_to_expiry', 'call_gex', 'put_gex', 'net_gex', 'abs_gex']
       expiry  days_to_expiry      call_gex       put_gex       net_gex  \
0  2026-04-30               0  2.956730e+09 -1.452948e+09  1.503782e+09   
1  2026-05-01               1  3.217459e+09 -1.442670e+09  1.774789e+09   

        abs_gex  
0  4.174261e+09  
1  3.238984e+09  


In [13]:
from yfinance_api3.classes.pricing import (
    ContractType, Space, ExerciseType
)
from yfinance_api3.classes.positions_book import (
    PositionsBook, PortfolioBook, WatchList
)

# ── Build INTC book ───────────────────────────────────────────────────────
book = PositionsBook(
    symbol         = "INTC",
    spot           = 93.22,
    vol            = 0.61,           # 61% hist vol
    risk_free_rate = 4.50,           # 4.50%
    div_yield      = 0.0,
    contract_type  = ContractType.STOCK,
    lot_size       = 100,
)

# add legs from your sheet (negative lots = short)
book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61, "equity")
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61, "equity")
book.add_option("call", "2026-01-30", 40,   0,  1.80, 0.54, "equity")
book.add_option("call", "2026-06-18", 40,   0,  4.50, 0.61, "equity")
book.add_option("call", "2026-06-18", 50,   0,  7.30, 0.61, "equity")
book.add_option("call", "2026-01-30", 37,   0, 10.40, 0.52, "equity")

# underlying position
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# ── Greeks table ──────────────────────────────────────────────────────────
print(book.greeks_table(days_ahead=0).to_string())

# ── Summary ───────────────────────────────────────────────────────────────
s = book.summary(days_ahead=0)
for k, v in s.items():
    print(f"  {k:20s}: {v}")

# ── +30 days simulation ───────────────────────────────────────────────────
print("\n--- +30 days ---")
s30 = book.summary(days_ahead=30)
for k, v in s30.items():
    print(f"  {k:20s}: {v}")

# ── Save ─────────────────────────────────────────────────────────────────
book.save("positions/INTC.json")
print("\nSaved to positions/INTC.json")

# ── Portfolio ─────────────────────────────────────────────────────────────
port = PortfolioBook(name="My portfolio")
port.add_book(book, path="positions/INTC.json")
print(port.summary())
port.save("positions/portfolio.json")

# ── WatchList ─────────────────────────────────────────────────────────────
wl = WatchList()
wl.add("INTC", notes="short calls position")
wl.add("SPY",  notes="hedge")
wl.save("positions/watchlist.json")

     leg_id         type      expiry strike     iv          model exercise   lots lot_size price_paid model_price      value        pnl     delta  gamma     vega   theta      rho
0  71d73dc1         Call  2026-12-18   85.0  61.0%  Black-Scholes        E    -60      100      18.11     22.7065 -136238.79  -27578.79  -4131.40 -46.78 -1576.34  237.92 -1581.98
1  dcb09e22         Call  2026-12-18   90.0  61.0%  Black-Scholes        E    -40      100       21.5     20.3813  -81525.03    4474.97  -2583.58 -32.82 -1105.76  165.01 -1012.64
2  1f3464a1         Call  2026-01-30   40.0  54.0%  Black-Scholes        E      0      100        1.8       53.22       0.00       0.00      0.00   0.00     0.00    0.00     0.00
3  d51259c6         Call  2026-06-18   40.0  61.0%  Black-Scholes        E      0      100        4.5     53.4611       0.00       0.00      0.00   0.00     0.00   -0.00     0.00
4  889cf646         Call  2026-06-18   50.0  61.0%  Black-Scholes        E      0      100        7.3    

In [15]:
from yfinance_api3.classes.pricing import Space, ContractType
from yfinance_api3.classes.positions_book import PositionsBook, PortfolioBook

# rebuild INTC book
book = PositionsBook(
    symbol="INTC", spot=93.22, vol=0.61,
    risk_free_rate=4.50, div_yield=0.0,
)
book.add_option("call", "2026-12-18", 85, -60, 18.11, 0.61)
book.add_option("call", "2026-12-18", 90, -40, 21.50, 0.61)
book.add_underlying(lots=10000, entry_price=30.0, direction="long")

# single ticker dashboard — replicates your sheet
#plots.positions_book(book, days_ahead=0).show()
plots.positions_book(book, days_ahead=30).show()   # +30 days simulation

# portfolio summary
port = PortfolioBook(name="My portfolio")
port.add_book(book)
plots.portfolio_summary(port, days_ahead=0).show()